# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"DOI: {getattr(metadata, 'identifier', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets
print("Available record sets (@id and name):")
record_sets = metadata.record_sets
if not record_sets or len(record_sets) == 0:
    print("No record sets found in metadata. The dataset may not include tabular record sets (e.g., may provide only files). Try to view dataset.distributions instead.")
else:
    for rs in record_sets:
        print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

    # For each record set, print its fields (columns)
    for rs in record_sets:
        print(f"\nRecord set: {rs['@id']} (name: {rs.get('name', '')}) fields:")
        for field in rs.get('fields', []):
            print(f"  - @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")

# If no record sets, examine distributions
if not getattr(metadata, 'record_sets', None):
    if getattr(metadata, 'distributions', None):
        print("\nAvailable distribution files (possibly tabular):")
        for d in metadata.distributions:
            print(f"- @id: {d['@id']}, name: {d.get('name', '')}, url: {d.get('contentUrl', d.get('content_url', ''))}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Attempt to extract data from the available record set(s)
# If there are no record_sets, the dataset likely implements only "distributions" (i.e., files)

dataframes = {}

if getattr(metadata, 'record_sets', None) and len(metadata.record_sets):
    # List all record sets by @id
    record_set_ids = [rs['@id'] for rs in metadata.record_sets]
    print(f"Record set ids: {record_set_ids}")
    
    for record_set_id in record_set_ids:
        # mlcroissant needs the exact @id for the record set
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns for record set {record_set_id}: {df.columns.tolist()}")
        display(df.head())
else:
    print("No Croissant record_sets found. Attempting to load the main tabular file from distributions...")
    if getattr(metadata, 'distributions', None) and len(metadata.distributions):
        # Try to find the main tabular data file (CSV or similar)
        tabular_distributions = [d for d in metadata.distributions if ('csv' in (d.get('encodingFormat', '') or '').lower())]
        if not tabular_distributions:
            tabular_distributions = metadata.distributions
        for dist in tabular_distributions:
            file_url = dist.get('contentUrl', dist.get('content_url', None))
            if file_url:
                df = pd.read_csv(file_url)
                dataframes[dist['@id']] = df
                print(f"Loaded distribution file: {file_url}")
                print(f"First columns of {dist['@id']}: {df.columns.tolist()}")
                display(df.head())
            else:
                print(f"Could not find a data file url for distribution {dist['@id']}")
    else:
        print("No distributions defining a data file found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section applies operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For this dataset, let us choose a numeric field, such as MW (molecular weight) if available.

# Identify first available dataframe
main_key = None
if dataframes:
    main_key = list(dataframes.keys())[0]
    df = dataframes[main_key]
    print(f"Using dataframe from record set/distribution: {main_key}")

    # Heuristically pick a likely numeric field
    # Try the following names in order
    candidate_fields = ['MW', 'molecular_weight', 'molecularWeight', 'Molecular_Weight', 'coverage', 'abundance', 'peptide_count', 'PeptideCount', 'Intensity']
    available_numeric_fields = [c for c in candidate_fields if c in df.columns]
    if not available_numeric_fields:
        # Otherwise, pick the first float column
        float_cols = df.select_dtypes(include=['float64', 'int64']).columns
        if len(float_cols):
            numeric_field = float_cols[0]
        else:
            numeric_field = df.columns[0]  # fallback
    else:
        numeric_field = available_numeric_fields[0]

    print(f"Chosen numeric field for EDA: {numeric_field}")

    # Filtering: keep only proteins with MW > threshold (e.g., 10 kDa)
    threshold = 10
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
    print(f"Filtered records where {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # If available, group by a likely categorical field (e.g., Description, Gene, SampleId)
    candidate_group_fields = ['description', 'Description', 'gene', 'Gene', 'SampleId', 'sample_id']
    group_field = [c for c in candidate_group_fields if c in df.columns]
    if group_field:
        group_field = group_field[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped (mean {numeric_field}) by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Using EDA code's variable selection
    if 'filtered_df' in locals() and len(filtered_df):
        # Histogram of numeric field
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[numeric_field].astype(float), kde=True)
        plt.title(f"Distribution of {numeric_field} (filtered)")
        plt.xlabel(numeric_field)
        plt.show()

        # Boxplot by possible group/categorical variable
        if 'group_field' in locals() and group_field:
            num_groups = len(filtered_df[group_field].unique())
            if num_groups>1 and num_groups<=20:  # not too many!
                plt.figure(figsize=(10, 4))
                sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
                plt.title(f"{numeric_field} by {group_field}")
                plt.xticks(rotation=45, ha='right')
                plt.show()
    else:
        print(f"No filtered data to plot for {numeric_field}.")
else:
    print("No dataframe for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Summary**:
- The FAIR² dataset contains measurements of protein properties and quantities from mass spectrometry of extracellular vesicles from human mast cells.
- We demonstrated how to load and inspect metadata using `mlcroissant`, and how to load, filter, normalize, and visualize key numeric fields (e.g., molecular weight).
- This notebook provides a starting point for further proteomics investigations, such as biomarker identification or comparative analyses across protein subgroups.

> For deeper biological interpretation, refer to dataset annotations and original experimental documentation referenced in the metadata.